In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
r = pd.read_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_nap_1800_100.csv", index_col=0)
print(r.shape)
r.head()

(29708, 282)


,slice_index,filename,start_frame,end_frame,_neutral_mean,browDownLeft_mean,browDownRight_mean,browInnerUp_mean,browOuterUpLeft_mean,browOuterUpRight_mean,...,blinks_median_duration,blinks_std_duration,blinks_min_duration,blinks_max_duration,naps_count,naps_mean_duration,naps_median_duration,naps_std_duration,naps_min_duration,naps_max_duration
0,0,2_21_10_30fps_blendshape.csv,0,1799,0.0,0.020961,0.011672,0.091767,0.175344,0.169944,...,5.0,2.604979,1,14,0,0.0,0.0,0.0,0,0
1,1,2_21_10_30fps_blendshape.csv,100,1899,0.0,0.023722,0.013317,0.084261,0.166261,0.161839,...,5.0,2.259955,1,14,0,0.0,0.0,0.0,0,0
2,2,2_21_10_30fps_blendshape.csv,200,1999,0.0,0.024122,0.013600,0.087417,0.166378,0.161378,...,5.0,2.309985,1,14,0,0.0,0.0,0.0,0,0
3,3,2_21_10_30fps_blendshape.csv,300,2099,0.0,0.024967,0.013989,0.085200,0.162722,0.157400,...,5.0,2.132263,1,14,0,0.0,0.0,0.0,0,0
4,4,2_21_10_30fps_blendshape.csv,400,2199,0.0,0.026233,0.014633,0.082817,0.159100,0.153994,...,5.0,2.070616,1,14,0,0.0,0.0,0.0,0,0


In [3]:
r['target'] = r['filename'].apply(lambda x:int(x.split('_')[2].replace('10','2').replace('0','0').replace('5','1')))
r['fold'] = r['filename'].apply(lambda x: int(x.split('_')[0]))
r['subject_id'] = r['filename'].apply(lambda x: int(x.split('_')[1]))
r['target'].value_counts()

2    10416
1     9782
0     9510
Name: target, dtype: int64

In [4]:

# Define the neural network model for ordinal regression
def build_ordinal_nn(input_shape):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))
    
    for u in units:
        model.add(layers.Dense(u, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_ratio))
    
    # Output layer for ordinal regression
    model.add(layers.Dense(1, activation=lambda x: 3 * tf.keras.activations.sigmoid(x) - 1.5))  
    
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-4),
        loss='mse',  # MSE is a common loss for ordinal regression
        metrics=['accuracy']
    )
    return model

# Define XGBoost model
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    xgb_params = {
        'objective': 'reg:squarederror',  # Using squared error loss for ordinal treatment
        'eval_metric': 'mae',
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 500,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    }
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    return model

# Define LightGBM model
def train_lightgbm(X_train, y_train, X_val, y_val, X_test, y_test):
    lgb_params = {
        'objective': 'regression_l1',  # L1 loss works well for ordinal data
        'metric': 'mae',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'n_estimators': 500
    }
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    return model

In [5]:
# def loss(y_true, y_pred):
#     weights = K.cast(K.abs(K.argmax(y_true, axis=1) - K.argmax(y_pred, axis=1))/(K.int_shape(y_pred)[1] - 1), dtype='float32')
#     return (1.0 + weights) * losses.categorical_crossentropy(y_true, y_pred)

def ordinal_accuracy(y_true, y_pred):
    # Define thresholds for classification
    thresholds = [0.33, 0.66]  # For mapping sigmoid to 0, 1, 2
    y_pred_class = np.digitize(y_pred, thresholds)  # Map sigmoid output to class labels (0, 1, 2)
    return tf.keras.metrics.sparse_categorical_accuracy(y_true, y_pred_class)

def step(val, steps, outputs):
    for i in range(len(steps)-1):
        low = steps[i]
        high = steps[i+1]
        if val >= low and val <= high:
            return outputs[i]
        
def report(model, x, y, output_dict):
    y_pred = np.round(model.predict(x))
#     y_pred = [step(v,[0,0.33,0.66,1],[0,1,2]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-1.5,-0.5,0.5,1.5],[-1,0,1]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-3,-1,1,3],[-2,0,2]) for v in model.predict(x, verbose=0)]
    if output_dict:
        cr = classification_report(list(y), y_pred, output_dict=True)
        cr['confusion_matrix'] = list(confusion_matrix(list(y), y_pred))
        return cr
    
    print(classification_report(list(y), y_pred))
    print(confusion_matrix(list(y), y_pred))

In [6]:
class Model:
    def __init__(self, max_depth, n_estimators, subsample, colsample_bytree, cv_on='subject_id'):
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.cv_on = cv_on
        self.models = []
        self.histories = []
        self.reports = []
    
    
    def get_reports(self):
        records = []

        for item in self.reports:
            base_info = {'test': item['test'], 'val': item['val']}
            for report_type in ['train_report', 'val_report', 'test_report']:
                report = item.get(report_type, {})
                for category, metrics in report.items():
                    if category == 'confusion_matrix':
                        cm_flat = np.array(metrics).flatten()
                        for i, value in enumerate(cm_flat):
                            records.append({**base_info, 'report_type': report_type, 'category': f'conf_matrix_{i}', 'metric': value})
                    elif isinstance(metrics, dict):
                        for metric, value in metrics.items():
                            records.append({**base_info, 'report_type': report_type, 'category': category, 'metric_name': metric, 'metric_value': value})
                    else:
                        records.append({**base_info, 'report_type': report_type, 'category': 'macro avg', 'metric_name': category, 'metric_value': metrics})

        return pd.DataFrame(records)

        
    
    def test_model(self, data):
        unique = data[self.cv_on].unique()
        for i in unique:
            for j in unique:
                if i==j:
                    continue
                    
                print("==============================")
                print(f"Test {i} & Validation {j}")
                
                train = data[~data[self.cv_on].isin([i,j])].reset_index(drop=True)
                val   = data[data[self.cv_on]==j].reset_index(drop=True)
                test  = data[data[self.cv_on]==i].reset_index(drop=True)
                
                X_train = train.drop(columns=['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id'])
                X_val   = val.drop(columns=['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id'])
                X_test  = test.drop(columns=['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id'])
                
                y_train = train['target']
                y_val   = val['target']
                y_test  = test['target']
                
                scaler  = StandardScaler().fit(X_train)
                
                X_train = scaler.transform(X_train)
                X_val   = scaler.transform(X_val)
                X_test  = scaler.transform(X_test)
                
                print(f"train dataset dimension {X_train.shape}")
                print(f"validation dataset dimension {X_val.shape}")
                print(f"test dataset dimension {X_test.shape}")
                
                xgb_params = {
                    'objective': 'reg:pseudohubererror',
                    'eval_metric': 'mae',
                    'max_depth': self.max_depth,
                    'learning_rate': 0.03,
                    'n_estimators': self.n_estimators,
                    'subsample': self.subsample,
                    'colsample_bytree': self.colsample_bytree,
                    'reg_lambda': 10,
                    'reg_alpha': 5
                }

                model = xgb.XGBRegressor(**xgb_params)
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
                
                self.models.append(model)
                
                train_report = report(model, X_train, y_train, True)
                val_report   = report(model, X_val, y_val, True)
                test_report  = report(model, X_test, y_test, True)
                
                self.reports.append({'test' : i, 
                                     'val' : j, 
                                     'train_report' : train_report, 
                                     'val_report' : val_report, 
                                     'test_report' : test_report})

In [7]:
m = Model(max_depth = 5, n_estimators = 200, subsample = 0.75, colsample_bytree = 0.75)
m.test_model(r)

Test 21 & Validation 4
train dataset dimension (28719, 278)
validation dataset dimension (499, 278)
test dataset dimension (490, 278)
Test 21 & Validation 7
train dataset dimension (28729, 278)
validation dataset dimension (489, 278)
test dataset dimension (490, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 21 & Validation 1
train dataset dimension (28730, 278)
validation dataset dimension (488, 278)
test dataset dimension (490, 278)
Test 21 & Validation 5
train dataset dimension (28726, 278)
validation dataset dimension (492, 278)
test dataset dimension (490, 278)
Test 21 & Validation 20
train dataset dimension (28676, 278)
validation dataset dimension (542, 278)
test dataset dimension (490, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 21 & Validation 39
train dataset dimension (28890, 278)
validation dataset dimension (328, 278)
test dataset dimension (490, 278)
Test 21 & Validation 50
train dataset dimension (28673, 278)
validation dataset dimension (545, 278)
test dataset dimension (490, 278)


...

/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 29 & Validation 32
train dataset dimension (28580, 278)
validation dataset dimension (650, 278)
test dataset dimension (478, 278)
Test 29 & Validation 17
train dataset dimension (28736, 278)
validation dataset dimension (494, 278)
test dataset dimension (478, 278)
Test 29 & Validation 30
train dataset dimension (28687, 278)
validation dataset dimension (543, 278)
test dataset dimension (478, 278)
Test 29 & Validation 13
train dataset dimension (28706, 278)
validation dataset dimension (524, 278)
test dataset dimension (478, 278)
Test 29 & Validation 18
train dataset dimension (28729, 278)
validation dataset dimension (501, 278)
test dataset dimension (478, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 29 & Validation 14
train dataset dimension (28714, 278)
validation dataset dimension (516, 278)
test dataset dimension (478, 278)
Test 29 & Validation 31
train dataset dimension (28703, 278)
validation dataset dimension (527, 278)
test dataset dimension (478, 278)
Test 29 & Validation 33
train dataset dimension (28757, 278)
validation dataset dimension (473, 278)
test dataset dimension (478, 278)
Test 29 & Validation 24
train dataset dimension (28610, 278)
validation dataset dimension (620, 278)
test dataset dimension (478, 278)


KeyboardInterrupt: 

In [ ]:
df = m.get_reports()
df[(df['metric_name'] == 'accuracy') & (df['report_type'] == 'train_report')]

In [ ]:
r = pd.read_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_7200_100.csv", index_col=0)
print(r.shape)
r['target'] = r['filename'].apply(lambda x:int(x.split('_')[2].replace('10','2').replace('0','0').replace('5','1')))
r['fold'] = r['filename'].apply(lambda x: int(x.split('_')[0]))
r['subject_id'] = r['filename'].apply(lambda x: int(x.split('_')[1]))
r['target'].value_counts()
m = Model(max_depth = 5, n_estimators = 200, subsample = 0.75, colsample_bytree = 0.75)
m.test_model(r)

(20275, 282)
Test 21 & Validation 4
train dataset dimension (19610, 278)
validation dataset dimension (337, 278)
test dataset dimension (328, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 21 & Validation 7
train dataset dimension (19620, 278)
validation dataset dimension (327, 278)
test dataset dimension (328, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 21 & Validation 1
train dataset dimension (19621, 278)
validation dataset dimension (326, 278)
test dataset dimension (328, 278)


...

Test 30 & Validation 12
train dataset dimension (19602, 278)
validation dataset dimension (292, 278)
test dataset dimension (381, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 30 & Validation 8
train dataset dimension (19567, 278)
validation dataset dimension (327, 278)
test dataset dimension (381, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 30 & Validation 3
train dataset dimension (19531, 278)
validation dataset dimension (363, 278)
test dataset dimension (381, 278)


In [ ]:
X_val.shape

In [ ]:
xgb_params = {
    'objective': 'reg:pseudohubererror',
    'eval_metric': 'mae',
    'max_depth': 5,
    'learning_rate': 0.03,
    'n_estimators': 600,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_lambda': 10,
    'reg_alpha': 5
}


In [37]:
model = xgb.XGBRegressor(**xgb_params)

In [38]:
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

[0]	validation_0-mae:0.72233
[1]	validation_0-mae:0.72273
[2]	validation_0-mae:0.72054
[3]	validation_0-mae:0.71826
[4]	validation_0-mae:0.71698
[5]	validation_0-mae:0.71128
[6]	validation_0-mae:0.70620
[7]	validation_0-mae:0.69599
[8]	validation_0-mae:0.69448
[9]	validation_0-mae:0.69609
[10]	validation_0-mae:0.69347
[11]	validation_0-mae:0.68879
[12]	validation_0-mae:0.68317
[13]	validation_0-mae:0.68431
[14]	validation_0-mae:0.68459
[15]	validation_0-mae:0.68703
[16]	validation_0-mae:0.68857
[17]	validation_0-mae:0.68854
[18]	validation_0-mae:0.68845
[19]	validation_0-mae:0.68883
[20]	validation_0-mae:0.68765
[21]	validation_0-mae:0.68947
[22]	validation_0-mae:0.68781
[23]	validation_0-mae:0.68710
[24]	validation_0-mae:0.68510
[25]	validation_0-mae:0.68374
[26]	validation_0-mae:0.68543
[27]	validation_0-mae:0.68461
[28]	validation_0-mae:0.68560
[29]	validation_0-mae:0.68682
[30]	validation_0-mae:0.68840
[31]	validation_0-mae:0.68835
[32]	validation_0-mae:0.68448
[33]	validation_0-ma

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=600, n_jobs=None,
             num_parallel_tree=None, objective='reg:pseudohubererror', ...)

In [39]:
print(classification_report(y_train, np.round(model.predict(X_train))))
print(confusion_matrix(y_train, np.round(model.predict(X_train))))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      9180
           1       0.95      1.00      0.97      9455
           2       1.00      0.97      0.98     10086

    accuracy                           0.98     28721
   macro avg       0.98      0.98      0.98     28721
weighted avg       0.98      0.98      0.98     28721

[[8974  206    0]
 [   2 9450    3]
 [   0  315 9771]]


In [40]:
print(classification_report(y_val, np.round(model.predict(X_val))))
print(confusion_matrix(y_val, np.round(model.predict(X_val))))

              precision    recall  f1-score   support

           0       0.90      0.05      0.10       167
           1       0.34      0.99      0.50       165
           2       0.00      0.00      0.00       167

    accuracy                           0.35       499
   macro avg       0.41      0.35      0.20       499
weighted avg       0.41      0.35      0.20       499

[[  9 158   0]
 [  1 164   0]
 [  0 167   0]]


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [41]:
print(classification_report(y_test, np.round(model.predict(X_test))))
print(confusion_matrix(y_test, np.round(model.predict(X_test))))

              precision    recall  f1-score   support

           0       0.24      0.26      0.25       163
           1       0.23      0.43      0.30       162
           2       1.00      0.07      0.13       163

    accuracy                           0.25       488
   macro avg       0.49      0.25      0.23       488
weighted avg       0.49      0.25      0.23       488

[[ 42 121   0]
 [ 92  70   0]
 [ 40 112  11]]


In [53]:
df = m.get_reports()
df[(df['metric_name'] == 'accuracy') & (df['report_type'] == 'test_report')]

,test,val,report_type,category,metric_name,metric_value,metric
72,1,4,test_report,macro avg,accuracy,0.153689,NaN
162,1,3,test_report,macro avg,accuracy,0.237705,NaN
252,1,2,test_report,macro avg,accuracy,0.108607,NaN
342,2,4,test_report,macro avg,accuracy,0.444219,NaN
432,2,1,test_report,macro avg,accuracy,0.444219,NaN
533,2,3,test_report,macro avg,accuracy,0.460446,NaN


In [52]:
df[(df['category'].isin(["conf_matrix_0", "conf_matrix_1", "conf_matrix_2", "conf_matrix_3", "conf_matrix_4", "conf_matrix_5", "conf_matrix_6", "conf_matrix_7", "conf_matrix_8"])) & (df['test']==21)]

,test,val,report_type,category,metric_name,metric_value,metric
